# Data Preparation

Prepares the Amazon Product Reviews 2014 dataset for the TIGER pipeline.

**Paper reference**: Rajput et al., "Recommender Systems with Generative Retrieval" (NeurIPS 2023)

**What this notebook does:**
1. Load ratings data (user interactions)
2. Filter to users with ≥ 5 reviews
3. Build user interaction sequences (last 20 items, ordered by time)
4. Leave-one-out train/val/test split
5. Load item metadata for items in our dataset
6. Format item text and generate embeddings with Sentence-T5

**Dataset**: Amazon Product Reviews 2014, Toys and Games category.  
Download from: https://cseweb.ucsd.edu/~jmcauley/datasets/amazon/links.html

The paper reports these stats for Toys and Games (Table 6):
| Stat | Value |
|------|-------|
| Users | 19,412 |
| Items | 11,924 |
| Mean sequence length | 8.63 |
| Median sequence length | 6 |

In [1]:
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm

In [2]:
# Constants
CACHE_DIR = Path("../.cache")
DATA_DIR = Path("../data/2014")
DATASET_NAME = "Toys_and_Games"
REVIEWS_FILE = (
    DATA_DIR / f"reviews_{DATASET_NAME}_5.json"
)  # 5-core: every user & item has >= 5 interactions
META_FILE = DATA_DIR / f"meta_{DATASET_NAME}.json"

MAX_SEQ_LEN = 20  # Paper: limit history to last 20 items

OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert REVIEWS_FILE.exists(), f"File not found: {REVIEWS_FILE}"
assert META_FILE.exists(), f"File not found: {META_FILE}"

## 1. Load Reviews (5-core)

The 5-core file is pre-filtered: every user and every item has at least 5 interactions. This is the standard preprocessing used by SASRec, BERT4Rec, S3-Rec, and TIGER.

The file is JSON (one object per line) with columns:
- `reviewerID` → our `user_id`
- `asin` → item ID
- `overall` → rating (1.0–5.0)
- `unixReviewTime` → Unix timestamp

We keep **all ratings** (1–5 stars) — the paper does not filter by rating.

In [3]:
df_reviews = pd.read_json(REVIEWS_FILE, lines=True)

# Rename columns to our standard names
df_reviews = df_reviews.rename(
    columns={
        "reviewerID": "user_id",
        "overall": "rating",
        "unixReviewTime": "timestamp",
    }
)[["user_id", "asin", "rating", "timestamp"]].copy()

print(f"Total reviews: {len(df_reviews):,}")
print(f"Unique users:  {df_reviews['user_id'].nunique():,}")
print(f"Unique items:  {df_reviews['asin'].nunique():,}")
df_reviews.head()

Total reviews: 167,597
Unique users:  19,412
Unique items:  11,924


,user_id,asin,rating,timestamp
0,A1VXOAVRGKGEAK,0439893577,5,1390953600
1,A8R62G708TSCM,0439893577,4,1395964800
2,A21KH420DK0ICA,0439893577,5,1359331200
3,AR29QK6HPFYZ4,0439893577,5,1391817600
4,ACCH8EOML6FN5,0439893577,4,1399248000


In [4]:
# Rating distribution
print("Rating distribution:")
print(df_reviews["rating"].value_counts().sort_index())
print()

# Verify 5-core property
min_user_reviews = df_reviews.groupby("user_id").size().min()
min_item_reviews = df_reviews.groupby("asin").size().min()
print(f"Min reviews per user: {min_user_reviews} (should be >= 5)")
print(f"Min reviews per item: {min_item_reviews} (should be >= 5)")

Rating distribution:
rating
1      4707
2      6298
3     16357
4     37445
5    102790
Name: count, dtype: int64

Min reviews per user: 5 (should be >= 5)
Min reviews per item: 5 (should be >= 5)


## 2. User & Item Statistics

The 5-core file is already filtered, no additional filtering needed. Let's verify the data matches the paper's expectations.

In [5]:
# No filtering needed — 5-core data already satisfies the paper's requirement.
# We just rename for consistency with downstream code.
df_filtered = df_reviews.copy()

print(f"Reviews: {len(df_filtered):,}")
print(f"Users:   {df_filtered['user_id'].nunique():,}")
print(f"Items:   {df_filtered['asin'].nunique():,}")

Reviews: 167,597
Users:   19,412
Items:   11,924


In [6]:
# Distribution of reviews per user
print("Reviews per user:")
print(df_filtered.groupby("user_id").size().describe())
print()
print("Reviews per item:")
print(df_filtered.groupby("asin").size().describe())

Reviews per user:
count    19412.000000
mean         8.633680
std          8.507474
min          5.000000
25%          5.000000
50%          6.000000
75%          9.000000
max        550.000000
dtype: float64

Reviews per item:
count    11924.000000
mean        14.055434
std         15.849858
min          5.000000
25%          6.000000
50%          9.000000
75%         15.000000
max        309.000000
dtype: float64


## 3. Build User Sequences

For each user, sort their interactions by timestamp and keep the **last 20** items. This becomes the input to the sequence-to-sequence model.

The paper (Section 3.2):
> "We construct item sequences for every user by sorting chronologically the items they have interacted with."

In [7]:
# Sort by user and timestamp
df_filtered = df_filtered.sort_values(["user_id", "timestamp"])

# Group by user, take last MAX_SEQ_LEN items
sequences = df_filtered.groupby("user_id")["asin"].apply(list)

print(f"Users with sequences: {len(sequences):,}")

seq_lengths = sequences.apply(len)
assert seq_lengths.min() >= 5
print(f"Mean sequence length:  {seq_lengths.mean():.2f}")
print(f"Median seq length:    {seq_lengths.median():.0f}")
print(f"Min / Max:            {seq_lengths.min()} / {seq_lengths.max()}")

Users with sequences: 19,412
Mean sequence length:  8.63
Median seq length:    6
Min / Max:            5 / 550


## 4. Leave-One-Out Split

Following the paper's evaluation protocol (Appendix C):
- **Last item** → test
- **Second-to-last** → validation
- **Rest** → training

Users with fewer than 3 items are dropped (need at least 1 train + 1 val + 1 test).

> "Following the standard evaluation protocol, we use the leave-one-out strategy for evaluation. For each item sequence, the last item is used for testing, the item before the last is used for validation, and the rest is used for training."

In [8]:
splits = []
for user_id, seq in sequences.items():
    # 5-core already guarantees len(seq) >= 5, but this protects split validity.
    if len(seq) < 3:
        continue
    splits.append(
        {
            "user_id": user_id,
            "train_items": seq[:-2],
            "val_item": seq[-2],
            "test_item": seq[-1],
        }
    )

df_splits = pd.DataFrame(splits)

print(f"Users in final dataset: {len(df_splits):,}")
print(f"Mean train seq length:   {df_splits['train_items'].apply(len).mean():.2f}")

Users in final dataset: 19,412
Mean train seq length:   6.63


In [9]:
# Collect all unique items in the final dataset
all_items = set()
for _, row in df_splits.iterrows():
    all_items.update(row["train_items"])
    all_items.add(row["val_item"])
    all_items.add(row["test_item"])

print(f"Unique items in final dataset: {len(all_items):,}")

Unique items in final dataset: 11,924


In [10]:
# Compare with paper's Table 6
print("Our stats vs Paper (Toys and Games):")
print(f"  Users:       {len(df_splits):,}  (paper: 19,412)")
print(f"  Items:       {len(all_items):,}  (paper: 11,924)")
print(f"  Mean seq len: {seq_lengths.mean():.2f}  (paper: 8.63)")
print(f"  Med seq len: {seq_lengths.median():.0f}  (paper: 6)")

Our stats vs Paper (Toys and Games):
  Users:       19,412  (paper: 19,412)
  Items:       11,924  (paper: 11,924)
  Mean seq len: 8.63  (paper: 8.63)
  Med seq len: 6  (paper: 6)


## 5. Load Item Metadata

The metadata file uses **"loose" JSON** — actually Python dict literals with single quotes, 
Python `True`/`False`/`None`, etc. The [dataset page](https://cseweb.ucsd.edu/~jmcauley/datasets/amazon/links.html) 
recommends using Python's `eval()` to parse it. We use `ast.literal_eval` which is the safe equivalent.

We only keep metadata for items that are in our final dataset — no wasted memory.

In [11]:
import ast

meta_records = []
skipped = 0
with open(META_FILE, "r", encoding="utf-8", errors="ignore") as f:
    for line in tqdm(f, desc="Parsing metadata"):
        try:
            record = ast.literal_eval(line.strip())
            if record.get("asin") in all_items:
                meta_records.append(record)
        except (ValueError, SyntaxError):
            skipped += 1
            continue

df_meta = pd.DataFrame(meta_records)
print(f"Items with metadata: {len(df_meta):,} / {len(all_items):,}")
print(f"Lines skipped (parse errors): {skipped:,}")
df_meta.head()

Parsing metadata: 336072it [00:20, 16090.44it/s]


Items with metadata: 11,924 / 11,924
Lines skipped (parse errors): 0


,asin,categories,description,title,price,imUrl,brand,related,salesRank
0,0439893577,"[[Toys & Games, Dress Up & Pretend Play, Prete...",The Magnetic Tabletop Learning Easel is one of...,Little Red Tool Box: Magnetic Tabletop Learnin...,16.19,http://ecx.images-amazon.com/images/I/51XCjcMt...,Scholastic,"{'also_bought': ['B0015KEB2M', 'B001ECI9EM', '...",NaN
1,048645195X,"[[Toys & Games, Arts & Crafts, Drawing & Paint...",,Dover Publications-Decorative Tile Designs Col...,1.44,http://ecx.images-amazon.com/images/I/61r4Gp10...,Dover Pubns,"{'also_bought': ['0486456943', '0486456420', '...","{'Arts, Crafts & Sewing': 1009}"
2,0545496470,"[[Toys & Games, Novelty & Gag Toys, Magic Kits...",,The Book of Impossible Objects: 25 Eye-Popping...,11.13,http://ecx.images-amazon.com/images/I/61mEDfXS...,Klutz,"{'also_bought': ['1570548943', '1452109036', '...","{'Arts, Crafts & Sewing': 1462}"
3,0615444172,"[[Toys & Games, Toy Remote Control & Play Vehi...",Our Original Sticker Books follow friends and ...,Original Sticker Book Starter Kit with 3 Puffy...,16.99,http://ecx.images-amazon.com/images/I/51ayc5M9...,NaN,"{'also_bought': ['B00GOXK692', '0985090901', '...",{'Toys & Games': 27977}
4,0670010936,"[[Toys & Games, Stuffed Animals & Plush, Anima...","Cuddle up with little llama llama, star ofLlam...",Llama Llama Plush,21.73,http://ecx.images-amazon.com/images/I/419pYIEQ...,NaN,"{'also_bought': ['0670059838', '0670016489', '...",{'Toys & Games': 313485}


In [12]:
# Check coverage of key fields
print("Metadata columns:", df_meta.columns.tolist())
print()
for col in ["title", "brand", "price", "description", "categories"]:
    if col in df_meta.columns:
        non_null = df_meta[col].notna().sum()
        print(
            f"  {col}: {non_null:,} / {len(df_meta):,} ({non_null/len(df_meta)*100:.0f}%)"
        )

Metadata columns: ['asin', 'categories', 'description', 'title', 'price', 'imUrl', 'brand', 'related', 'salesRank']

  title: 11,865 / 11,924 (100%)
  brand: 10,204 / 11,924 (86%)
  price: 11,080 / 11,924 (93%)
  description: 11,746 / 11,924 (99%)
  categories: 11,924 / 11,924 (100%)


## 6. Format Item Text for Embeddings

The paper (Section 4) uses:
> "item's content features such as title, price, brand, and category to construct a sentence, which is then passed to the pre-trained Sentence-T5 model to obtain the item's semantic embedding of 768 dimension."

We combine these fields into a single text string.

In [13]:
def format_item_text(row: pd.Series) -> str:
    """Combine item metadata into a single text string for embedding.

    Following the paper: "item's content features such as title, price,
    brand, and category to construct a sentence" (Section 4).
    """
    parts = []

    if pd.notna(row.get("title")):
        parts.append(str(row["title"]))

    if pd.notna(row.get("description")):
        desc = row["description"]
        if isinstance(desc, list):
            desc = ". ".join(str(d) for d in desc if d)
        if desc:
            parts.append(str(desc))

    if pd.notna(row.get("brand")):
        parts.append(f"Brand: {row['brand']}")

    categories = row.get("categories")
    if isinstance(categories, list) and categories:
        # categories is list of lists, e.g. [['Toys & Games', 'Puzzles']]
        flat = [c for sublist in categories for c in sublist if c]
        if flat:
            parts.append("Category: " + " > ".join(flat))

    if pd.notna(row.get("price")) and row["price"]:
        parts.append(f"Price: ${row['price']}")

    return "\n".join(parts) if parts else "Unknown product"

In [14]:
df_meta["text"] = df_meta.apply(format_item_text, axis=1)

# Show a few examples
for _, row in df_meta.sample(3, random_state=42).iterrows():
    print(f"ASIN: {row['asin']}")
    print(f"Text: {row['text'][:300]}")
    print("---")

ASIN: B007J3FA8I
Text: Fisher-Price Imaginext DC Super Friends Gotham City Jail
Imagine   One of Gotham City's biggest criminals, Bane, in Gotham City Jail, when, with the turn of a disk, he "powers up" and begins to glow. That cant be good for Batman. Turn a figure on another disk and&#x2014;jailbreak&#x2014;Bane is free
---
ASIN: B000FMQ3AY
Text: Snap Circuits Motion Detector
Elenco's Snap Circuits makes learning electronics easy and fun! Just follow the colorful pictures in our manual and build exciting projects such as AM radios, burglar alarms, doorbells and much more! You can even play electronic games with your friends. All parts are mo
---
ASIN: B001TH8LCW
Text: Incan Gold: Quest for Riches in the Ruins
Incan Gold (co-designed by Alan R. Moon, designer of Ticket to Ride) is a quick, fun game of bluff and daring in which explorers push their luck while exploring an old Incan temple in search of gold and treasure. In each round, you decide whether to delve de
---


## 7. Generate Embeddings with Sentence-T5

The paper uses **Sentence-T5** (`sentence-t5-base`, 768-dim) to encode item text.

> "We use the pre-trained Sentence-T5 model to obtain the semantic embedding of each item in the dataset." (Section 4)

Reference: Ni et al., "Sentence-T5: Scalable sentence encoders from pre-trained text-to-text models" (ACL 2022).

For ~12K items this takes a few minutes.

In [15]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer(
    "sentence-transformers/sentence-t5-base", cache_folder=CACHE_DIR
)
print(f"Model loaded: {st_model.get_sentence_embedding_dimension()} dimensions")

Loading weights:   0%|          | 0/99 [00:00<?, ?it/s]

Model loaded: 768 dimensions


In [16]:
texts = df_meta["text"].tolist()
asins = df_meta["asin"].tolist()

raw_embeddings = st_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=128,
    convert_to_tensor=True,
)
print(f"Embeddings shape: {raw_embeddings.shape}")

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Embeddings shape: torch.Size([11924, 768])


In [17]:
# Create a dict mapping asin -> embedding tensor
embeddings = {asin: emb for asin, emb in zip(asins, raw_embeddings.cpu())}
print(
    f"Embeddings: {len(embeddings):,} items, {next(iter(embeddings.values())).shape[0]} dims"
)

# Storage estimate
total_bytes = len(embeddings) * 768 * 4
print(f"Estimated storage: {total_bytes / 1e6:.1f} MB")

Embeddings: 11,924 items, 768 dims
Estimated storage: 36.6 MB


## 8. Save Processed Data

Save everything we need for the next phases:
- `splits.parquet` — user sequences with train/val/test split
- `items.parquet` — item metadata with formatted text
- `embeddings.pt` — item embeddings (asin → 768-dim tensor)

In [18]:
# Save splits
df_splits.to_parquet(OUTPUT_DIR / "splits.parquet", index=False)
print(f"Saved splits: {len(df_splits):,} users")

# Save item metadata with text
items_to_save = df_meta[["asin", "text"]].copy()
items_to_save.to_parquet(OUTPUT_DIR / "items.parquet", index=False)
print(f"Saved items: {len(items_to_save):,} items")

# Save embeddings
torch.save(embeddings, OUTPUT_DIR / "embeddings.pt")
print(f"Saved embeddings: {len(embeddings):,} items")

print(f"\nAll data saved to {OUTPUT_DIR}/")

Saved splits: 19,412 users
Saved items: 11,924 items
Saved embeddings: 11,924 items

All data saved to ../data/2014/processed/


In [19]:
# Final summary
print("=" * 50)
print("DATA PREPARATION COMPLETE")
print("=" * 50)
print(f"Category:         {DATASET_NAME}")
print(f"Users:            {len(df_splits):,}")
print(f"Items (total):    {len(all_items):,}")
print(f"Items with meta:  {len(df_meta):,}")
print(f"Embeddings:       {len(embeddings):,}")
print(f"Mean seq len:     {seq_lengths.mean():.2f}")
print(f"Embedding dim:    {st_model.get_sentence_embedding_dimension()}")
print()
print("Paper (Table 6):")
print("  Users:          19,412")
print("  Items:          11,924")
print("  Mean seq len:   8.63")

DATA PREPARATION COMPLETE
Category:         Toys_and_Games
Users:            19,412
Items (total):    11,924
Items with meta:  11,924
Embeddings:       11,924
Mean seq len:     8.63
Embedding dim:    768

Paper (Table 6):
  Users:          19,412
  Items:          11,924
  Mean seq len:   8.63
